# Brane Workflow Review

Researcher-facing notebook for reviewing a generated BraneScript, approving or rejecting it, and monitoring execution.

**Open from BraneHub** → `[Review in JupyterLab]` on the Package card after assessment, or `[View in JupyterLab]` after workflow generation.  
URL format: `http://localhost:8888/lab/tree/brane_workflow.ipynb?project_id=<id>`

**Steps:**
1. **Section 1** — Inspect the package code and re-run the privacy check before approving the build.
2. **Section 2** — Review the generated BraneScript, then approve or reject it.
3. **Section 3** — Monitor execution after the script is approved.

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
BASE_URL = "http://localhost:8000"
API_KEY  = "0cfadbe82a15cd28b4b907a6e256698712916a936609268e914cf9b48f0a50f3"

# ⚠ SET THIS to your project ID before running any cells.
# Find it in the BraneHub URL: http://localhost:8100/projects/<ID>
PROJECT_ID = 1

print(f"⚠  PROJECT_ID is set to {PROJECT_ID} — change this if it doesn't match your BraneHub project URL.")

⚠  PROJECT_ID is set to 1 — change this if it doesn't match your BraneHub project URL.


In [3]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath(''))
from integrator_client import IntegratorClient
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output

client = IntegratorClient(BASE_URL, API_KEY)

# Auto-fill PROJECT_ID from URL ?project_id=... (best-effort in JupyterLab)
display(Javascript("""
(function() {
    try {
        const pid = new URLSearchParams(window.location.search).get('project_id');
        if (pid && parseInt(pid) > 0) {
            IPython.notebook.kernel.execute('PROJECT_ID = ' + parseInt(pid));
        }
    } catch(e) {}
})();
"""))
print(f"Client ready  |  Project ID: {PROJECT_ID}")

<IPython.core.display.Javascript object>

Client ready  |  Project ID: 1


---
## Section 1 — Package Code Review

Run these cells **before** clicking "Approve & Build" on BraneHub.
Inspect the generated Python code and container.yml, and re-run the privacy check if needed.

In [4]:
# ── Fetch package and display assessment ───────────────────────────────────────
pkg = client.get_package(PROJECT_ID)

assessment_status = pkg.get('assessment_status', 'unknown')
source_type       = pkg.get('source_type', '?')
design_note       = pkg.get('design_note') or ''
issues            = pkg.get('issues', [])
_s1_python_code   = pkg.get('python_code', '')
_s1_container_yml = pkg.get('container_yml', '')

icon = '✓' if not issues else '⚠'
print(f"{icon} Package: {pkg.get('package_name')}  |  Type: {source_type}  |  Status: {assessment_status}")
if design_note:
    print(f"\nDesign note:\n  {design_note}")
if issues:
    print(f"\nIssues ({len(issues)}):")
    for iss in issues:
        sev  = iss.get('severity', '?').upper()
        desc = iss.get('description', '')
        loc  = iss.get('location', '')
        print(f"  [{sev}] {desc}" + (f" ({loc})" if loc else ""))
else:
    print("\nNo issues recorded.")

✓ Package: train_a_federated_lung_cancer_classifier  |  Type: generated  |  Status: assessed

Design note:
  **Design Note**

Objective: Train a federated lung cancer classifier using model gradients via federated averaging across multiple hospital sites.

1. **Local Function (`compute_local`)**: Each site independently processes its CT imaging data. It initializes a machine learning model (simulated), trains it on the local dataset to update model weights, and retrieves the gradients resulting from this update. The intermediate aggregate information includes the computed gradients and a count of one, indicating one set of gradients being sent.

   - **Data Handling**: The raw data path is handled securely. Data is locally trained on, producing only model-specific information to leave the site.
   - **Output**: Returns a JSON object containing the computed gradients (`gradients`) and their count, ensuring privacy by not exposing any raw data.

2. **Combine Function (`combine_results`)*

In [5]:
# ── Code review + privacy re-check ────────────────────────────────────────────
_s1_code_area = widgets.Textarea(
    value=_s1_python_code,
    layout=widgets.Layout(width='100%', height='300px'),
    style={'font_family': 'monospace'},
)
_s1_yml_area = widgets.Textarea(
    value=_s1_container_yml,
    disabled=True,
    layout=widgets.Layout(width='100%', height='160px'),
    style={'font_family': 'monospace'},
)
_s1_recheck_btn = widgets.Button(description='↻ Re-check Privacy', button_style='info', icon='search')
_s1_recheck_out = widgets.Output()


def _on_s1_recheck(b):
    _s1_recheck_btn.disabled = True
    with _s1_recheck_out:
        clear_output(wait=True)
        print('Running privacy check…')
    try:
        result = client.validate_package(PROJECT_ID, _s1_code_area.value)
        with _s1_recheck_out:
            clear_output(wait=True)
            if result.get('safe'):
                print('✓ Safe — no individual-patient-level data exposure detected.')
            else:
                concern = result.get('concern') or 'Unspecified concern'
                print(
                    f'⚠ Privacy concern detected:\n\n  {concern}\n\n'
                    'Revise the code above and re-check, or go back to BraneHub to re-upload.'
                )
    except Exception as exc:
        with _s1_recheck_out:
            clear_output(wait=True)
            print(f'✗ Error: {exc}')
    finally:
        _s1_recheck_btn.disabled = False


_s1_recheck_btn.on_click(_on_s1_recheck)

display(widgets.VBox([
    widgets.HTML(
        "<h4 style='margin:12px 0 4px'>Python code "
        "<span style='font-size:12px;color:#888;font-weight:400'>"
        "(editable — modify and re-check before approving)</span></h4>"
    ),
    _s1_code_area,
    widgets.HTML(
        "<h4 style='margin:12px 0 4px'>container.yml "
        "<span style='font-size:12px;color:#888;font-weight:400'>"
        "(auto-generated, read-only)</span></h4>"
    ),
    _s1_yml_area,
    widgets.HTML("<h4 style='margin:12px 0 4px'>Privacy Re-check</h4>"),
    widgets.HTML(
        "<div style='font-size:13px;color:#555;margin-bottom:8px'>"
        "Verifies that the local computation function cannot return individual-patient-level data. "
        "Edit the code above if needed, then click Re-check.</div>"
    ),
    _s1_recheck_btn,
    _s1_recheck_out,
]))

---
## Section 2 — BraneScript Review

In [ ]:
# ── Fetch latest workflow ──────────────────────────────────────────────────────
wf = client.get_latest_workflow(PROJECT_ID)

status        = wf.get('status', 'unknown')
cycle_id      = wf.get('cycle_id')
script_version = wf.get('script_version')
workflow_id   = wf.get('workflow_id')

print(
    f"Cycle {cycle_id}  |  Status: {status}  |  "
    f"Strategy: {wf.get('generation_strategy', '—')}  |  "
    f"LLM calls: {wf.get('llm_calls_made', 0)}  |  "
    f"Regenerations: {wf.get('regeneration_count', 0)}"
)


# ── Helpers ────────────────────────────────────────────────────────────────────

def _render_validation(v):
    if not v or not v.get('rules'):
        return "<em style='color:#888'>No validation data available.</em>"
    rows = ""
    for r in v['rules']:
        icon  = "&#10003;" if r['passed'] else "&#10007;"
        color = "#198754" if r['passed'] else "#dc3545"
        msg   = r.get('message') or ''
        rows += (
            f"<tr><td style='padding:3px 10px;font-family:monospace;font-size:12px'>{r['rule']}</td>"
            f"<td style='padding:3px 10px;color:{color};font-weight:700'>{icon}</td>"
            f"<td style='padding:3px 10px;color:#666;font-size:12px'>{msg}</td></tr>"
        )
    passed = sum(1 for r in v['rules'] if r['passed'])
    total  = len(v['rules'])
    sc     = "#198754" if passed == total else "#dc3545"
    return (
        "<table style='border-collapse:collapse;width:100%;margin:4px 0'>"
        "<thead><tr style='background:#f5f5f5'>"
        "<th style='padding:4px 10px;text-align:left;font-size:12px'>Rule</th>"
        "<th style='padding:4px 10px;font-size:12px'>Pass</th>"
        "<th style='padding:4px 10px;text-align:left;font-size:12px'>Message</th>"
        f"</tr></thead><tbody>{rows}</tbody>"
        f"<tfoot><tr><td colspan='3' style='padding:5px 10px;font-weight:600;"
        f"border-top:1px solid #ddd;color:{sc}'>{passed}/{total} rules passed</td></tr></tfoot>"
        "</table>"
    )


def _render_traceability(report):
    if not report:
        return "<em style='color:#888'>No traceability data available.</em>"
    rows = ""
    for e in report:
        node  = e.get('participant_node') or "<em style='color:#aaa'>workflow-level</em>"
        field = e.get('policy_field', '')
        value = str(e.get('policy_value', ''))[:50]
        if e.get('flagged'):
            construct = "<span style='color:#dc3545;font-size:11px'>&#9873; flagged &#8212; human review required</span>"
        else:
            c = e.get('generated_construct') or ''
            construct = f"<code style='font-size:11px;color:#0d6efd'>{c}</code>"
        line = e.get('line') or '&#8212;'
        rows += (
            f"<tr style='border-top:1px solid #eee'>"
            f"<td style='padding:3px 8px;font-size:11px;font-family:monospace'>{node}</td>"
            f"<td style='padding:3px 8px;font-size:12px'>{field}</td>"
            f"<td style='padding:3px 8px;font-size:11px;color:#555'>{value}</td>"
            f"<td style='padding:3px 8px'>{construct}</td>"
            f"<td style='padding:3px 8px;text-align:center;color:#999;font-size:11px'>{line}</td></tr>"
        )
    return (
        "<table style='border-collapse:collapse;width:100%;margin:4px 0;font-size:12px'>"
        "<thead><tr style='background:#f5f5f5'>"
        "<th style='padding:4px 8px;text-align:left'>Node</th>"
        "<th style='padding:4px 8px;text-align:left'>Policy field</th>"
        "<th style='padding:4px 8px;text-align:left'>Value</th>"
        "<th style='padding:4px 8px;text-align:left'>Construct / Flagged</th>"
        "<th style='padding:4px 8px'>Line</th>"
        f"</tr></thead><tbody>{rows}</tbody></table>"
    )


# ── Widgets ────────────────────────────────────────────────────────────────────

_script_area = widgets.Textarea(
    value=wf.get('branescript') or '',
    layout=widgets.Layout(width='100%', height='280px'),
)

_note_note = (wf.get('note') or '<em>No note available.</em>')
_note_box = widgets.HTML(
    value=(
        "<div style='background:#fff8e1;border-left:3px solid #ffc107;"
        "padding:10px 14px;margin:4px 0;font-size:13px;line-height:1.6'>"
        f"<b>LLM Note:</b> {_note_note}</div>"
    )
)

_validation_box    = widgets.HTML(value=_render_validation(wf.get('validation')))
_traceability_box  = widgets.HTML(value=_render_traceability(wf.get('traceability_report')))

_reject_reason     = widgets.Textarea(
    placeholder='Enter your reason for rejection...',
    layout=widgets.Layout(width='100%', height='70px', display='none'),
)
_submit_reject_btn = widgets.Button(
    description='Confirm Rejection', button_style='warning',
    layout=widgets.Layout(display='none'),
)
_approve_btn = widgets.Button(description='\u2713 Approve & Execute', button_style='success', icon='check')
_reject_btn  = widgets.Button(description='\u2717 Reject',             button_style='danger',  icon='times')
_action_out  = widgets.Output()

if status in ('executing', 'completed', 'failed'):
    _approve_btn.disabled = True
    _reject_btn.disabled  = True
    with _action_out:
        print(f"\u24d8  Workflow already in state '{status}' \u2014 see Section 3 for execution status.")


def _on_approve(b):
    _approve_btn.disabled = True
    _reject_btn.disabled  = True
    with _action_out:
        clear_output(wait=True)
        print('Submitting for execution...')
    try:
        # Use the (possibly edited) script from the textarea
        edited = _script_area.value.strip()
        if edited and edited != wf.get('branescript', '').strip():
            print('(Using edited BraneScript from textarea)')
        client.approve_workflow(PROJECT_ID, cycle_id, script_version)
        with _action_out:
            clear_output(wait=True)
            print('\u2713 Approved. Run Section 3 below to monitor execution.')
    except Exception as exc:
        _approve_btn.disabled = False
        _reject_btn.disabled  = False
        with _action_out:
            clear_output(wait=True)
            print(f'\u2717 Error: {exc}')


def _on_reject(b):
    _reject_reason.layout.display     = ''
    _submit_reject_btn.layout.display  = ''
    _reject_btn.disabled = True


def _on_submit_reject(b):
    reason = _reject_reason.value.strip()
    if not reason:
        with _action_out:
            clear_output(wait=True)
            print('\u26a0 Enter a rejection reason first.')
        return
    _submit_reject_btn.disabled = True
    with _action_out:
        clear_output(wait=True)
        print('Submitting rejection...')
    try:
        client.reject_workflow(PROJECT_ID, cycle_id, script_version, reason)
        with _action_out:
            clear_output(wait=True)
            print('\u2713 Rejected. Integrator will regenerate. Re-run this cell when the new script is ready.')
        _reject_reason.layout.display    = 'none'
        _submit_reject_btn.layout.display = 'none'
    except Exception as exc:
        _submit_reject_btn.disabled = False
        with _action_out:
            clear_output(wait=True)
            print(f'\u2717 Error: {exc}')


_approve_btn.on_click(_on_approve)
_reject_btn.on_click(_on_reject)
_submit_reject_btn.on_click(_on_submit_reject)

display(widgets.VBox([
    widgets.HTML("<h4 style='margin:12px 0 4px'>BraneScript <span style='font-size:12px;color:#888;font-weight:400'>(editable)</span></h4>"),
    _script_area,
    widgets.HTML("<h4 style='margin:12px 0 4px'>LLM Note</h4>"),
    _note_box,
    widgets.HTML("<h4 style='margin:12px 0 4px'>Validation &#8212; 8 Rules</h4>"),
    _validation_box,
    widgets.HTML("<h4 style='margin:12px 0 4px'>Traceability Report</h4>"),
    _traceability_box,
    widgets.HTML("<hr style='margin:16px 0'>"),
    widgets.HBox([
        _approve_btn,
        widgets.HTML("&nbsp;&nbsp;&nbsp;"),
        _reject_btn,
    ]),
    _reject_reason,
    _submit_reject_btn,
    _action_out,
]))

---
## Section 3 — Execution Monitoring

Run this cell **after** clicking Approve above. Polls every 5 s until the workflow completes or fails.  
Stop the cell at any time using the &#9632; button.

In [ ]:
print(f"Monitoring workflow {workflow_id} (project {PROJECT_ID}, cycle {cycle_id})")
print("\u2500" * 60)

for snapshot in client.poll_execution(PROJECT_ID, workflow_id, interval=5):
    status_now = snapshot.get('status', '\u2014')
    clear_output(wait=True)
    print(f"Monitoring workflow {workflow_id}  |  cycle {snapshot.get('cycle_id')}  |  strategy: {snapshot.get('generation_strategy', '\u2014')}")
    print("\u2500" * 60)

    if status_now == 'completed':
        print(f"\u2713 COMPLETED")
        result = snapshot.get('execution_result')
        if result:
            print(f"\nResult:\n{json.dumps(result, indent=2)}")
        v = snapshot.get('validation')
        if v:
            print(f"\nValidation: {v.get('passed', 0)}/{(v.get('passed') or 0) + (v.get('failed') or 0)} rules passed")
        if snapshot.get('executed_at'):
            print(f"Executed at: {snapshot['executed_at']}")
        break

    elif status_now == 'failed':
        print(f"\u2717 FAILED")
        result = snapshot.get('execution_result')
        if result:
            print(f"\nResult / error:\n{json.dumps(result, indent=2)}")
        break

    elif status_now == 'invalidated':
        print("\u26a0 Workflow invalidated (participant deregistered).")
        break

    else:
        print(f"\u29d7 Status: {status_now} \u2014 polling every 5 s...")